# 🧪 Tahap 3: Exploratory Data Analysis (EDA) & Business Value
Analisis fundamental terhadap pendapatan finansial Time Laundry dari dua sudut pandang:
- **Level Paket Transaksi (Combo-Level)**: Berdasarkan apa yang diketik kasir utuh di nota.
- **Level Satuan Barang (Item-Level)**: Variasi dipecah per item, dikalikan harga satuan.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

sns.set_theme(style='whitegrid')

# Load Data
df_cleaned = pd.read_csv('../data/mba-laundry-transaction-20260704.csv')
df_price_ref = pd.read_csv('../data/mba-laundry-pricelist-20260704.csv')

# Cleaning
if 'Unnamed: 6' in df_cleaned.columns:
    df_cleaned = df_cleaned.drop(columns=['Unnamed: 6'])
if df_cleaned['Kg'].dtype == 'object':
    df_cleaned['Kg'] = df_cleaned['Kg'].str.replace(',', '.').astype(float)

# Helper
def format_rupiah(val):
    return f"Rp {int(val):,.0f}".replace(',', '.')

print('Data siap untuk EDA!')

## 3.1. Analisis Pendapatan Level Paket Transaksi (Combo-Level)
Menganalisis pendapatan riil berdasarkan string variasi utuh di nota kasir.

In [ ]:
# Tabel Top 10 Pendapatan per Paket Transaksi
data_transaksi = df_cleaned.groupby('Variasi')['Total harga'].sum().sort_values(ascending=False).reset_index()
data_tabel = data_transaksi.head(10).copy()
data_tabel['Total harga'] = data_tabel['Total harga'].apply(format_rupiah)

print('Tabel Top 10 Pendapatan Berdasarkan Paket Transaksi Utuh:')
display(data_tabel)

💡 **Pembahasan:**
- Paket **Cuci Kering, Setrika** mendominasi dengan omset lebih dari Rp 4 juta.
- Ini menunjukkan bahwa layanan mencuci dan menyetrika adalah kombinasi paling diminati pelanggan.

In [ ]:
# Grafik Bar Chart - Pendapatan Level Paket Transaksi
plt.figure(figsize=(16, 9))
ax = sns.barplot(data=data_transaksi, x='Variasi', y='Total harga', color='#00796B')

plt.title('Perkiraan Total Pendapatan per Varian (Pendapatan Tertinggi)', fontsize=16, fontweight='bold')
plt.ylabel('Total Pendapatan (Rp)', fontsize=12)
plt.xlabel('Variasi Layanan', fontsize=12)
plt.xticks(rotation=90, ha='center')
plt.tight_layout()
plt.show()

## 3.2. Analisis Pendapatan Level Satuan Barang (Item-Level)
Membongkar keranjang belanja, lalu menghitung potensi pendapatan per item berdasarkan harga satuan.

In [ ]:
# Pecah variasi menjadi item tunggal
all_items = []
for variasi in df_cleaned['Variasi']:
    items = [item.strip() for item in str(variasi).split(',')]
    all_items.extend(items)

item_counts = Counter(all_items)
price_map = dict(zip(df_price_ref['Variasi'], df_price_ref['Harga']))

records = []
for item, count in item_counts.items():
    harga = price_map.get(item, 0)
    records.append({'Layanan': item, 'Frekuensi': count, 'Harga_Satuan': harga, 'Total_Pendapatan': count * harga})

df_deskripsi = pd.DataFrame(records).sort_values('Total_Pendapatan', ascending=False).reset_index(drop=True)

# Tabel dengan format Rupiah
df_tabel = df_deskripsi.head(10).copy()
df_tabel['Harga_Satuan'] = df_tabel['Harga_Satuan'].apply(format_rupiah)
df_tabel['Total_Pendapatan'] = df_tabel['Total_Pendapatan'].apply(lambda x: format_rupiah(x))

print('Tabel Top 10 Pendapatan per Satuan Layanan:')
display(df_tabel)

💡 **Pembahasan:**
- Jika keranjang dibongkar secara matematis, **Setrika** muncul paling sering karena hampir selalu menjadi pendamping layanan lain.
- Secara omset premium, **Karpet** dan **Bed Cover** menjadi penyumbang terbesar karena harga satuannya tinggi.

In [ ]:
# Grafik Bar Chart - Pendapatan Level Satuan Barang
data_item = df_deskripsi.head(10)

plt.figure(figsize=(14, 7))
ax = sns.barplot(data=data_item, x='Layanan', y='Total_Pendapatan', color='#00796B')

for i, row in data_item.iterrows():
    ax.text(i, row['Total_Pendapatan'] + row['Total_Pendapatan'] * 0.01,
            format_rupiah(row['Total_Pendapatan']),
            ha='center', va='bottom', fontsize=8, fontweight='bold')

plt.title('Top 10 Pendapatan per Satuan Layanan (Item-Level)', fontsize=14, fontweight='bold')
plt.ylabel('Total Pendapatan (Rp)', fontsize=11)
plt.xlabel('Nama Layanan', fontsize=11)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

💡 **Pembahasan:**
- Grafik ini menunjukkan bahwa manajemen laundry tidak boleh hanya mengejar kuantitas baju.
- Layanan premium seperti Karpet dan Bed Cover perlu mendapat perhatian khusus dalam strategi pemasaran.